# Huấn luyện mô hình nhận diện biển số (ALPR) cho ValoParking

**HƯỚNG DẪN:**
1. Truy cập **[Google Colab](https://colab.research.google.com/)**, tải file này lên.
2. Chọn **Runtime -> Change runtime type** (Thời gian chạy -> Thay đổi loại thời gian chạy).
3. Ở mục **Hardware accelerator** (Bộ tăng tốc phần cứng), chọn **T4 GPU** rồi lưu lại.
4. Nhấn nút Play ở từng khối code (cell) bên dưới để chạy.

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install ultralytics roboflow

In [ ]:
from roboflow import Roboflow
from ultralytics import YOLO
import shutil
import os

print("Các thư viện đã sẵn sàng!")

### 2. Tải Dataset (Dữ liệu biển số xe)
Ở đây chúng ta sử dụng một bộ dataset public trên Roboflow. Bạn có thể tự thay đổi API Key và Project Name nếu bạn có dataset riêng.

In [ ]:
# Tải Dataset biển số xe Việt Nam (Public Dataset từ Roboflow)
# Lưu ý: api_key dưới đây là public key cho dataset mẫu. Nếu lỗi, bạn hãy vào roboflow.com lấy 1 dataset miễn phí.
rf = Roboflow(api_key="t3oG1X6wT6gS7hS1f0G9") # Thay thế bằng API Key của bạn nếu có
project = rf.workspace("lpr-k0qgq").project("license-plate-recognition-j3eqv")
dataset = project.version(1).download("yolov8")

print("Đã tải xong Dataset về thư mục:", dataset.location)

### 3. Bắt đầu Train AI (Huấn luyện)
Bước này sẽ mất khoảng 15-30 phút tùy thuộc vào lượng ảnh. Chúng ta dùng YOLOv8n (bản nhẹ nhất, chạy cực nhanh trên backend web).

In [ ]:
from google.colab import files
import os

# Tải file best.pt cũ lên (Transfer Learning)
print("Hãy nhấn Choose Files để chọn file best.pt từ máy tính của bạn tải lên Colab:")
uploaded = files.upload()

if os.path.exists("best.pt"):
    print("\nĐang tải mô hình từ kiến thức cũ (best.pt)...")
    model = YOLO("best.pt") # Dùng file model cũ thay vì yolov8n.pt
else:
    print("\nKhông tìm thấy best.pt, sẽ train lại từ đầu với yolov8n.pt...")
    model = YOLO("yolov8n.pt")

# Bắt đầu quá trình train với data mới (Tăng vòng lặp epochs lên 50)
results = model.train(data=f"{dataset.location}/data.yaml", epochs=50, imgsz=640, plots=True)

print("Huấn luyện bổ sung hoàn tất!")


### 4. Lấy file model `best.pt`
Sau khi train xong, mô hình xịn nhất sẽ được lưu ở file `best.pt`. Code bên dưới sẽ tải nó về máy tính của bạn.

In [ ]:
from google.colab import files

best_model_path = '/content/runs/detect/train/weights/best.pt'

if os.path.exists(best_model_path):
    print(f"Đang tải file {best_model_path} về máy...")
    files.download(best_model_path)
    print("Tải xong! Hãy copy file này vào thư mục 'ai_service' trong dự án ValoParking của bạn.")
else:
    print("Lỗi: Không tìm thấy file best.pt. Có thể quá trình train chưa hoàn thành hoặc bị lỗi.")